# VCARE CC Compass: tool evaluation (Option 1, "practice room")

**What this notebook does, in plain terms:** an AI assistant gets the three VCARE CC Compass tools and takes a 10-question test. Every question has a known, double-checked answer. The assistant has to pick the right tool, use it correctly and reply with the answer. Each answer is then graded and the scorecard is saved in MLflow.

**Analogy:** a new employee takes a practice exam in a training room down the hall, using an identical copy of the office toolbox. Nobody needs a security badge because they never leave the building. That's why this option needs no app sign-in setup.

**The three tools being tested**
| Tool | What it does |
|---|---|
| `ask_vcare_case_data` | Answers questions about the 600 synthetic consults, using the Genie agent |
| `get_consult_pathway_checklist` | Returns the GEMBA observation checklist and policy sources for each pathway step |
| `flag_high_risk_case` | Screens one synthetic case against the timeliness and patient-safety rules |

**How each answer is graded (three checks)**
1. **Right tool used:** did the assistant use the tool that question needs?
2. **Exact final answer:** does its final line match the answer key exactly?
3. **Judged correct:** an independent AI grader confirms the answer contains the correct fact (MLflow's `Correctness` judge).

**Goal (pass mark):** all 10 judged correct, the right tool used on all 10, and at least 9 of 10 exact final answers.

**Data:** 100% synthetic. No PHI or PII. No result here is a clinical determination.

**To run:** use *Run all*. It takes about 3 minutes. Gold questions: `eval/phoenix/datasets/mcp_vcare_eval_0/evaluation.xml`.

In [0]:
# Installs the server's libraries, then restarts Python so they load cleanly.
# MLflow (the scorecard tool) comes with Databricks; only a minimum version is required.
%pip install -q "mlflow>=3.4" "fastmcp>=3.0,<5" "pydantic-settings>=2.0" "databricks-sdk>=0.60" "openai>=1.40"
%restart_python

In [0]:
# Settings for this run. Change these values, not the code below.
import os

REPO = "/Workspace/Users/anna.fernandez@va.gov/mcp-vcare-compass"   # the Git folder this notebook lives in
GOLD_SET = f"{REPO}/eval/phoenix/datasets/mcp_vcare_eval_0/evaluation.xml"  # the 10 questions and answer key
EXPERIMENT = "/Users/anna.fernandez@va.gov/mcp_vcare_compass_eval"      # where scorecards are saved
AGENT_MODEL = "databricks-claude-sonnet-4-5"   # the AI assistant taking the test
GENIE_SPACE_ID = "01f1d47843d015a58a851ae41d4c422f"  # "Healthcare Risk and Coordination Analysis" Genie agent
RUN_NAME = "mcp_vcare_eval_0 option1"
PARALLEL_QUESTIONS = 2   # how many questions are worked on at once

# Which tool each question needs (question number -> tool). Used for the "right tool used" check.
EXPECTED_TOOL = {
    1: "ask_vcare_case_data", 2: "ask_vcare_case_data", 3: "ask_vcare_case_data",
    4: "ask_vcare_case_data", 5: "ask_vcare_case_data",
    6: "get_consult_pathway_checklist", 7: "get_consult_pathway_checklist",
    8: "flag_high_risk_case", 9: "flag_high_risk_case", 10: "flag_high_risk_case",
}

# The assistant's standing instructions: use only the tools, answer only.
SYSTEM_PROMPT = (
    "You answer questions about VA Community Care consult review using ONLY the provided tools. "
    "Call the tools you need, then reply with ONLY the final answer in the exact format the question "
    "asks for: no explanation, no working, no extra words."
)

os.environ["GENIE_SPACE_ID"] = GENIE_SPACE_ID          # read by the server's settings at import
os.environ["MLFLOW_GENAI_EVAL_MAX_WORKERS"] = str(PARALLEL_QUESTIONS)

In [0]:
# Loads the MCP server straight from this repo's code. This is the copy of the toolbox in the practice room.
import sys, xml.etree.ElementTree as ET
import pandas as pd

sys.path.insert(0, f"{REPO}/src")
from mcp_vcare_compass import __version__ as SERVER_VERSION
from mcp_vcare_compass.app import mcp
from mcp_vcare_compass.config import settings

assert settings.genie_space_id, "GENIE_SPACE_ID is not set (see Step 2)"

gold = [
    {"q": i, "question": p.findtext("question"), "answer": p.findtext("answer"), "tool": EXPECTED_TOOL[i]}
    for i, p in enumerate(ET.parse(GOLD_SET).getroot().findall("qa_pair"), start=1)
]
assert len(gold) == 10, f"expected 10 questions, found {len(gold)}"
print(f"Server: {mcp.name} v{SERVER_VERSION} | questions loaded: {len(gold)}")
display(pd.DataFrame(gold)[["q", "tool", "answer", "question"]])

In [0]:
# The assistant sees only the three tools: their names, descriptions and input formats.
# It decides which tool to call; we run the call against the server and pass the result back.
# Every question is recorded as an MLflow "trace" (a step-by-step log), so you can replay what happened.
import asyncio, json, threading
import mlflow
from mlflow.entities import SpanType
from fastmcp import Client
from databricks.sdk import WorkspaceClient

mlflow.set_tracking_uri("databricks")
experiment = mlflow.set_experiment(EXPERIMENT)
mlflow.openai.autolog()   # records every call to the AI model inside the trace

# The MCP client is async. One background event loop is shared by every question,
# so parallel questions never fight over loops.
_LOOP = asyncio.new_event_loop()
threading.Thread(target=lambda: (asyncio.set_event_loop(_LOOP), _LOOP.run_forever()), daemon=True).start()
def run_async(coro):
    return asyncio.run_coroutine_threadsafe(coro, _LOOP).result()

async def _call_tool(name, args):
    async with Client(mcp) as client:            # in-memory connection to the server; no network or sign-in
        result = await client.call_tool(name, args)
        if result.structured_content is not None:
            return json.dumps(result.structured_content)
        return result.content[0].text

TOOLS = [
    {"type": "function", "function": {"name": t.name, "description": t.description, "parameters": t.parameters}}
    for t in run_async(mcp.list_tools())
]
llm = WorkspaceClient().serving_endpoints.get_open_ai_client()

@mlflow.trace(span_type=SpanType.AGENT)
def answer_question(question: str) -> str:
    """Let the assistant work the question with the tools, then return its final reply."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]
    for _ in range(8):   # at most 8 rounds of tool use per question
        msg = llm.chat.completions.create(model=AGENT_MODEL, messages=messages, tools=TOOLS, temperature=0).choices[0].message
        if not msg.tool_calls:
            return (msg.content or "").strip()
        messages.append({"role": "assistant", "content": msg.content or "",
                         "tool_calls": [tc.model_dump() for tc in msg.tool_calls]})
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            with mlflow.start_span(name=tc.function.name, span_type=SpanType.TOOL) as span:
                span.set_inputs(args)
                try:
                    output = run_async(_call_tool(tc.function.name, args))
                except Exception as e:   # a tool error is shown to the assistant, not raised
                    output = json.dumps({"error": str(e)[:500]})
                span.set_outputs(output[:4000])
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": output[:12000]})
    return "(no final answer within 8 rounds)"

print("Tools available to the assistant:", [t["function"]["name"] for t in TOOLS])
print("Scorecards are saved to experiment:", EXPERIMENT, f"(id {experiment.experiment_id})")

In [0]:
# 1) right tool used   2) exact final answer   3) judged correct by an independent AI grader
import re
from mlflow.entities import Feedback
from mlflow.genai.scorers import Correctness, scorer

def _normalize(text):
    """Compare letters and digits only (ignore case, spaces and punctuation)."""
    return re.sub(r"[^a-z0-9]", "", (text or "").lower())

@scorer
def used_expected_tool(trace, expectations) -> Feedback:
    used = [span.name for span in trace.search_spans(span_type=SpanType.TOOL)]
    ok = expectations["expected_tool"] in used
    return Feedback(value=ok, rationale=f"Tools called: {used or 'none'}. Needed: {expectations['expected_tool']}.")

@scorer
def final_answer_exact(outputs, expectations) -> Feedback:
    lines = [line for line in str(outputs).splitlines() if line.strip()]
    final_line = lines[-1] if lines else ""
    ok = _normalize(final_line) == _normalize(expectations["gold_answer"])
    return Feedback(value=ok, rationale=f"Final line {final_line!r} vs answer key {expectations['gold_answer']!r}.")

SCORERS = [used_expected_tool, final_answer_exact, Correctness()]   # Correctness checks expected_facts

# MLflow evaluation rows: the question goes to answer_question(question=...); the answer key is under expectations.
eval_rows = [
    {"inputs": {"question": g["question"]},
     "expectations": {"expected_facts": [g["answer"]], "gold_answer": g["answer"], "expected_tool": g["tool"]}}
    for g in gold
]
print(f"{len(eval_rows)} questions ready, {len(SCORERS)} checks each")

In [0]:
# Runs all 10 questions through the assistant and grades them. Takes about 3 minutes.
import time

started = time.time()
with mlflow.start_run(run_name=RUN_NAME) as run:
    mlflow.log_params({"agent_model": AGENT_MODEL, "server_version": SERVER_VERSION,
                       "gold_set": "mcp_vcare_eval_0", "questions": len(eval_rows)})
    results = mlflow.genai.evaluate(data=eval_rows, predict_fn=answer_question, scorers=SCORERS)
elapsed = round(time.time() - started)
RUN_ID = run.info.run_id
print(f"Finished in {elapsed} s | MLflow run id: {RUN_ID}")
print({name: round(float(value), 3) for name, value in results.metrics.items()})

In [0]:
# Builds a plain table: question, answer key, the assistant's answer and each check's result.
traces = mlflow.search_traces(run_id=RUN_ID, return_type="list")

def _check(trace, name):
    for a in trace.info.assessments:
        if a.name == name and getattr(a, "feedback", None) is not None:
            v = a.feedback.value
            return v in (True, "yes", "Yes", 1)
    return None

by_question = {g["question"]: g for g in gold}
rows = []
for t in traces:
    req = t.data.request
    question = (json.loads(req) if isinstance(req, str) else req).get("question")
    g = by_question.get(question)
    if g is None:
        continue
    resp = t.data.response
    answer = json.loads(resp) if isinstance(resp, str) and resp.startswith('"') else resp
    rows.append({"q": g["q"], "tool needed": g["tool"], "answer key": g["answer"], "assistant's answer": answer,
                 "right tool": _check(t, "used_expected_tool"), "exact answer": _check(t, "final_answer_exact"),
                 "judged correct": _check(t, "correctness")})
scorecard = pd.DataFrame(rows).sort_values("q").reset_index(drop=True)
assert len(scorecard) == 10, f"expected 10 graded questions, found {len(scorecard)}"
display(scorecard)

In [0]:
# Compares the scorecard with the pass mark set at the top of this notebook.
right_tool = int(scorecard["right tool"].sum())
exact = int(scorecard["exact answer"].sum())
correct = int(scorecard["judged correct"].sum())
checks = {
    "Right tool used on all 10": right_tool == 10,
    "Judged correct on all 10": correct == 10,
    "Exact final answer on at least 9 of 10": exact >= 9,
}
print(f"Right tool used: {right_tool}/10 | Exact final answer: {exact}/10 | Judged correct: {correct}/10\n")
for label, ok in checks.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {label}")
GOAL_MET = all(checks.values())
print("\nGOAL MET: the tools are ready for the live-app test (Option 2)." if GOAL_MET
      else "\nGOAL NOT MET: see the FAIL rows in the scorecard above and the traces in MLflow.")
mlflow.log_metrics({"goal_met": float(GOAL_MET), "right_tool_count": right_tool,
                    "exact_answer_count": exact, "judged_correct_count": correct}, run_id=RUN_ID)
print(f"\nSaved to MLflow experiment {EXPERIMENT}, run {RUN_ID}")

## How to read the results

- **Scorecard (Step 7):** one row per question. `True` means the check passed.
- **Goal check (Step 8):** PASS or FAIL against the pass mark at the top of the notebook.
- **Traces in MLflow:** open the experiment `/Users/anna.fernandez@va.gov/mcp_vcare_compass_eval`, then the run, then *Traces*. You can replay each question step by step: what the assistant asked, which tool it used, what the tool returned and what it answered.

**If a check fails, where the fix usually goes**
| Failed check | Likely cause | Where to fix |
|---|---|---|
| Right tool used | Tool description unclear | Docstring in `src/mcp_vcare_compass/tools/` |
| Exact final answer | Wordy reply, but right answer | `SYSTEM_PROMPT` in Step 2 |
| Judged correct | Tool returned wrong or incomplete data | The tool's logic, or the Genie agent's instructions |

**Next:** once the goal is met here, Option 2 repeats the same test against the live app (`vcare-pathway-checklist`). That also proves the app's sign-in and data permissions work.